# Step 8: Cost-Sensitive XGBoost 3-Variant Benchmark & Risk Policy Framework
**MSc Data Science Thesis — University of Wolverhampton**

###  Objective & Methodological Design
Train and evaluate **3 progressive XGBoost model variants** on the 80/20 data split ($N_{\text{train}} = 88,591$, $N_{\text{test}} = 22,148$) established in Step 6.5:

1. **Variant 1 (Cost-Unaware XGBoost)**: Standard XGBoost without sample cost weighting or threshold tuning ($t = 0.50$).
2. **Variant 2 (Cost-Aware XGBoost)**: XGBoost trained with Elkan sample cost weights ($w_i = \text{Cost}(FN_i) / \text{Mean}$) and default threshold ($t = 0.50$).
3. **Variant 3 (Cost-Aware XGBoost + OOF Threshold)**: XGBoost trained with sample cost weights AND 5-Fold Out-of-Fold (OOF) cross-validation decision threshold tuning ($t_{\text{opt}} = 0.65$).


In [1]:
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

print('======================================================================')
print('STEP 8: COST-SENSITIVE XGBOOST 3-VARIANT ABLATION EXPERIMENT')
print('======================================================================')

# Load dataset
df = pd.read_csv('data_with_cost_matrix.csv')
print(f'Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')

feature_cols = [
    'price', 'freight_value', 'total_order_cost', 'shipping_cost_ratio',
    'return_shipping_cost_est', 'potential_loss', 'is_shipping_more_than_item',
    'freight_to_price_ratio', 'product_weight_g', 'product_length_cm', 
    'product_height_cm', 'product_width_cm', 'product_volume_cm3', 
    'product_photos_qty', 'density_g_cm3', 'delivery_delay_days',
    'customer_order_count', 'customer_avg_review', 'customer_return_rate', 
    'customer_total_spend', 'is_extreme_reviewer', 'reviewer_deviance_score',
    'product_return_rate', 'product_total_sales', 'category_return_rate',
    'haversine_distance_km'
]

le = LabelEncoder()
if 'product_category_name_english' in df.columns:
    df['category_encoded'] = le.fit_transform(df['product_category_name_english'].fillna('unknown'))
    feature_cols.append('category_encoded')

feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df['is_returned']
w = df['sample_cost_weight'] if 'sample_cost_weight' in df.columns else None

# Stratified 80/20 train/test split (Matching Method 2 from Step 6.5)
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.20, random_state=42, stratify=y
)
test_indices = X_test.index
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

def compute_financial_loss(y_true, y_pred, df_full, idx_subset):
    fn_mask = (y_true == 1) & (y_pred == 0)
    fp_mask = (y_true == 0) & (y_pred == 1)
    fn_loss = df_full.loc[idx_subset[fn_mask], 'cost_FN'].sum()
    fp_loss = df_full.loc[idx_subset[fp_mask], 'cost_FP'].sum()
    return fn_loss + fp_loss, fn_mask.sum(), fp_mask.sum()

results = []

# --- Variant 1: Cost-Unaware XGBoost (Default t=0.50) ---
print('\n--- Training Variant 1: Cost-Unaware XGBoost (Default t=0.50) ---')
xgb1 = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=1.0, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
xgb1.fit(X_train, y_train)
y_pred1 = xgb1.predict(X_test)
y_prob1 = xgb1.predict_proba(X_test)[:, 1]
loss1, fn1, fp1 = compute_financial_loss(y_test, y_pred1, df, test_indices)
results.append({
    'Variant': 'Variant 1: Cost-Unaware XGBoost',
    'Sample Weights?': 'No', 'Threshold Strategy': 'Default (t=0.50)', 'Threshold (t)': 0.50,
    'Accuracy': f'{accuracy_score(y_test, y_pred1)*100:.2f}%',
    'Precision': f'{precision_score(y_test, y_pred1, zero_division=0)*100:.2f}%',
    'Recall': f'{recall_score(y_test, y_pred1)*100:.2f}%',
    'F1-Score': f'{f1_score(y_test, y_pred1, zero_division=0)*100:.2f}%',
    'AUC-ROC': f'{roc_auc_score(y_test, y_prob1)*100:.2f}%',
    'Total Loss (R$)': f'R$ {loss1:,.2f}', 'Loss Per Order': f'R$ {loss1/len(y_test):.4f}'
})

# --- Variant 2: Cost-Aware XGBoost (Default t=0.50) ---
print('\n--- Training Variant 2: Cost-Aware XGBoost (Default t=0.50) ---')
xgb2 = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
xgb2.fit(X_train, y_train, sample_weight=w_train)
y_pred2 = xgb2.predict(X_test)
y_prob2 = xgb2.predict_proba(X_test)[:, 1]
loss2, fn2, fp2 = compute_financial_loss(y_test, y_pred2, df, test_indices)
results.append({
    'Variant': 'Variant 2: Cost-Aware XGBoost',
    'Sample Weights?': 'Yes', 'Threshold Strategy': 'Default (t=0.50)', 'Threshold (t)': 0.50,
    'Accuracy': f'{accuracy_score(y_test, y_pred2)*100:.2f}%',
    'Precision': f'{precision_score(y_test, y_pred2, zero_division=0)*100:.2f}%',
    'Recall': f'{recall_score(y_test, y_pred2)*100:.2f}%',
    'F1-Score': f'{f1_score(y_test, y_pred2, zero_division=0)*100:.2f}%',
    'AUC-ROC': f'{roc_auc_score(y_test, y_prob2)*100:.2f}%',
    'Total Loss (R$)': f'R$ {loss2:,.2f}', 'Loss Per Order': f'R$ {loss2/len(y_test):.4f}'
})

# --- Variant 3: Cost-Aware XGBoost (5-Fold OOF CV Threshold) ---
print('\n--- Training Variant 3: Cost-Aware XGBoost (5-Fold OOF CV Threshold) ---')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs_xgb = np.zeros(len(X_train))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    w_tr = w_train.iloc[tr_idx] if w_train is not None else None
    spw = (y_tr == 0).sum() / (y_tr == 1).sum()
    m_cv = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
    if w_tr is not None:
        m_cv.fit(X_tr, y_tr, sample_weight=w_tr)
    else:
        m_cv.fit(X_tr, y_tr)
    oof_probs_xgb[val_idx] = m_cv.predict_proba(X_va)[:, 1]

train_indices = X_train.index
best_t_xgb = 0.50
min_oof_loss = float('inf')
for t in np.arange(0.05, 0.95, 0.01):
    y_pred_oof = (oof_probs_xgb >= t).astype(int)
    fn_mask = (y_train == 1) & (y_pred_oof == 0)
    fp_mask = (y_train == 0) & (y_pred_oof == 1)
    loss = df.loc[train_indices[fn_mask], 'cost_FN'].sum() + df.loc[train_indices[fp_mask], 'cost_FP'].sum()
    if loss < min_oof_loss:
        min_oof_loss = loss
        best_t_xgb = t

print(f'Optimal OOF Threshold for XGBoost Variant 3: {best_t_xgb:.2f}')
y_pred3 = (y_prob2 >= best_t_xgb).astype(int)
loss3, fn3, fp3 = compute_financial_loss(y_test, y_pred3, df, test_indices)
results.append({
    'Variant': 'Variant 3: Cost-Aware XGBoost + OOF Threshold',
    'Sample Weights?': 'Yes', 'Threshold Strategy': f'OOF Tuned (t={best_t_xgb:.2f})', 'Threshold (t)': round(best_t_xgb, 2),
    'Accuracy': f'{accuracy_score(y_test, y_pred3)*100:.2f}%',
    'Precision': f'{precision_score(y_test, y_pred3, zero_division=0)*100:.2f}%',
    'Recall': f'{recall_score(y_test, y_pred3)*100:.2f}%',
    'F1-Score': f'{f1_score(y_test, y_pred3, zero_division=0)*100:.2f}%',
    'AUC-ROC': f'{roc_auc_score(y_test, y_prob2)*100:.2f}%',
    'Total Loss (R$)': f'R$ {loss3:,.2f}', 'Loss Per Order': f'R$ {loss3/len(y_test):.4f}'
})

res_df = pd.DataFrame(results)
print('\n======================================================================')
print('STEP 8: XGBOOST 3-VARIANT RESULTS SUMMARY')
print('======================================================================')
print(res_df.to_string(index=False))
res_df.to_csv('cost_sensitive_results.csv', index=False)
print("\n✅ Saved 3-variant XGBoost benchmark results to 'cost_sensitive_results.csv'!")


STEP 8: COST-SENSITIVE XGBOOST 3-VARIANT ABLATION EXPERIMENT
Loaded dataset: 110,739 rows x 59 columns

--- Training Variant 1: Cost-Unaware XGBoost (Default t=0.50) ---

--- Training Variant 2: Cost-Aware XGBoost (Default t=0.50) ---

--- Training Variant 3: Cost-Aware XGBoost (5-Fold OOF CV Threshold) ---
Optimal OOF Threshold for XGBoost Variant 3: 0.65

STEP 8: XGBOOST 3-VARIANT RESULTS SUMMARY
                                      Variant Sample Weights? Threshold Strategy  Threshold (t) Accuracy Precision Recall F1-Score AUC-ROC Total Loss (R$) Loss Per Order
              Variant 1: Cost-Unaware XGBoost              No   Default (t=0.50)           0.50   90.06%    83.63% 45.20%   58.68%  80.47%   R$ 106,343.49      R$ 4.8015
                Variant 2: Cost-Aware XGBoost             Yes   Default (t=0.50)           0.50   85.33%    52.79% 56.97%   54.80%  79.83%   R$ 112,402.29      R$ 5.0751
Variant 3: Cost-Aware XGBoost + OOF Threshold             Yes OOF Tuned (t=0.65)        